# 03. 직접 찍은 제품 사진으로 정상/불량 영역 탐지 — 제조 현장 적용 가능성 검증

**시나리오** — 검사대 위에 여러 제품이 놓여 있다. 카메라 한 대로 **제품마다 위치(박스)와 상태(정상/불량)를 동시에** 판정하고,
한 판에 불량이 하나라도 있으면 NG로 처리한다.

**목표**
1. 직접 촬영·라벨링한 데이터로 YOLO를 학습해 정상/불량 영역이 제대로 표시되는지 확인한다.
2. **test 1** — 기본 성능: 박스 지표(mAP)와 판 단위 OK/NG 판정 정확도
3. **test 2** — 현장 조건 변화(어두움/밝음/흐림/노이즈)에서도 버티는지
4. 웹캠 실시간 판정으로 실제 라인 적용 가능성을 확인한다.

| 단계 | 스크립트 (레포 `src/`) |
|---|---|
| 촬영 | `capture.py` |
| 박스 라벨링 | `label.py` |
| train/val 분할 | `split.py` |
| 학습 | `train.py` |
| 평가 | `evaluate.py` |
| 실시간 판정 | `live.py` |

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from common import RAW_IMG_DIR, RAW_LBL_DIR, DATASET_DIR, BEST_WEIGHTS, RESULTS_DIR, REPORT_IMG_DIR, CLASSES, list_images, label_path
PY = sys.executable   # 이 커널의 파이썬(.venv)으로 스크립트 실행

## 1. 데이터 수집 — 웹캠 촬영
아래 셀을 실행하면 카메라 창이 뜬다. **space = 저장, q = 종료**

촬영 팁
- 한 장에 정상·불량 제품을 **섞어서 2~6개** 놓는다. 위치·각도·간격을 매번 바꾼다.
- 조명, 배경도 몇 가지로 바꿔 찍는다 (test 2 에서 조건 변화에 버티는 데 도움).
- 최소 **60~100장**, 불량 박스가 클래스별 **100개 이상**이면 좋다.

In [ ]:
!"{PY}" ../src/capture.py

## 2. 박스 라벨링
**드래그 = 박스**, `1`/`2` = normal/defect 선택, **우클릭 = 클래스 뒤집기**, `z` = 되돌리기, `d`/space = 다음, `a` = 이전, `q` = 종료

In [ ]:
!"{PY}" ../src/label.py

In [ ]:
imgs = list_images(RAW_IMG_DIR)
labeled = [p for p in imgs if label_path(p).exists()]
boxes = pd.Series([CLASSES[int(l.split()[0])] for p in labeled for l in label_path(p).read_text().splitlines() if l.strip()])
print(f"사진 {len(imgs)}장 / 라벨 완료 {len(labeled)}장")
boxes.value_counts().rename("boxes").to_frame()

**의견** — (실행 후 작성) 정상/불량 박스 수가 한쪽으로 치우쳤는지 확인. 치우쳤다면 적은 쪽을 더 찍는다.

## 3. train / val 분할

In [ ]:
!"{PY}" ../src/split.py --val 0.2

## 4. 학습
COCO 사전학습 `yolo11n.pt`에서 시작한다. 데이터가 적어서 에폭은 넉넉히(50) 주고, 15 에폭 동안 개선이 없으면 조기 종료한다.

In [ ]:
!"{PY}" ../src/train.py --epochs 50

In [ ]:
run_dir = BEST_WEIGHTS.parent.parent
display(Image.open(run_dir / "results.png"))

**의견** — (실행 후 작성) loss가 계속 줄어드는지, val mAP가 어디서 멈추는지, 과적합 징후가 있는지.

## 5. test 1 — 기본 성능

두 가지 관점으로 본다.
- **박스 지표**: 클래스별 Precision / Recall / mAP — "영역을 얼마나 정확히 찾나"
- **판 단위 판정**: 사진 한 장 = 제품 한 판. 불량 박스가 하나라도 있으면 NG
  - `missed_NG`(미검출) — 불량인데 OK로 통과. **제조에서 가장 치명적**
  - `false_NG`(과검출) — 정상인데 NG로 버림. 비용 문제

In [ ]:
!"{PY}" ../src/evaluate.py test1

In [ ]:
m1 = json.loads((RESULTS_DIR / "test1_metrics.json").read_text(encoding="utf-8"))
display(pd.DataFrame(m1["per_class"]).T)
display(pd.Series(m1["verdict"], name="verdict").to_frame())
display(Image.open(REPORT_IMG_DIR / "train_confusion_matrix.png"))

In [ ]:
Image.open(REPORT_IMG_DIR / "test1_samples.png")

In [ ]:
# 틀린 사진만 모아보기
per_img = pd.read_csv(RESULTS_DIR / "test1_per_image.csv")
per_img[~per_img.count_correct]

**test 1 분석** — (실행 후 작성)

## 6. 추가 실험 — 신뢰도 임계값에 따른 미검출/과검출
임계값을 낮추면 불량을 덜 놓치지만 정상을 불량으로 잘못 잡을 수 있다. 라인 운영 기준을 정하는 실험이다.

In [ ]:
from ultralytics import YOLO
import evaluate as ev

model = YOLO(str(BEST_WEIGHTS))
rows = []
for c in [0.1, 0.25, 0.4, 0.5, 0.6, 0.75]:
    ev.VERDICT_CONF = c
    s = ev.verdict_summary(ev.verdicts(model, DATASET_DIR / "images" / "val", DATASET_DIR / "labels" / "val"))
    rows.append({"conf": c, **s})
ev.VERDICT_CONF = 0.5
thr = pd.DataFrame(rows); thr.to_csv(RESULTS_DIR / "conf_sweep.csv", index=False)

ax = thr.plot(x="conf", y=["missed_NG", "false_NG"], marker="o", figsize=(7, 4), color=["#ef4444", "#f59e0b"])
ax.set_ylabel("images"); ax.set_title("confidence threshold vs missed / false NG")
plt.tight_layout(); plt.savefig(REPORT_IMG_DIR / "conf_sweep.png", dpi=150); plt.show()
thr

**의견** — (실행 후 작성) 미검출 0을 유지하면서 과검출이 가장 적은 임계값 → `live.py --conf` 에 사용

## 7. test 2 — 현장 조건 변화에 대한 강건성
검증 사진에 인위적으로 조건 변화를 준다. 조명 불량(dark/bright), 초점 흐림·진동(blur), 저조도 센서 노이즈(noise).

In [ ]:
!"{PY}" ../src/evaluate.py test2

In [ ]:
display(pd.read_csv(RESULTS_DIR / "test2_robustness.csv"))
display(Image.open(REPORT_IMG_DIR / "test2_robustness.png"))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 7))
for ax, cond in zip(axes, ["dark", "bright", "blur", "noise"]):
    ax.imshow(Image.open(REPORT_IMG_DIR / f"test2_{cond}.png")); ax.set_title(cond); ax.axis("off")
plt.tight_layout(); plt.show()

**test 2 분석** — (실행 후 작성) 어떤 조건에서 성능이 가장 떨어지는지, 현장 적용 시 조명·카메라 고정 등 어떤 대책이 필요한지.

## 8. 실시간 판정 데모
카메라 창에서 제품마다 초록(정상) / 빨강(불량) 박스가 표시되고, 상단에 판 전체 **OK/NG**가 나온다. `s` = 화면 저장, `q` = 종료

In [ ]:
!"{PY}" ../src/live.py --conf 0.5

In [ ]:
lives = sorted(RESULTS_DIR.glob("live_*.jpg"))
if lives:
    fig, axes = plt.subplots(1, min(3, len(lives)), figsize=(15, 5), squeeze=False)
    for ax, p in zip(axes[0], lives[-3:]):
        ax.imshow(Image.open(p)); ax.set_title(p.name); ax.axis("off")
    plt.tight_layout(); plt.show()

## 9. 결론 — 제조 현장 적용 가능성
- (실행 후 작성) 정확도, 미검출/과검출, 처리 속도(FPS) 기준으로 평가
- 실제 라인에 적용하려면: 조명·카메라 위치 고정, 불량 유형별 데이터 추가, 임계값 운영 기준, 주기적 재학습